# 00. Обзор проекта и исследовательский план

Этот ноутбук фиксирует общую логику проекта магистерской работы: от предметной области и датасета до production pipeline, backend и frontend.

Тема: **«Разработка веб-сервиса для прогнозирования успешности продуктов киноиндустрии»**.

Основной фокус: прогнозирование коммерческого успеха фильма через прогноз кассовой выручки.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

from pathlib import Path

important_paths = {
    "Основной очищенный датасет": PROJECT_ROOT / "data" / "output.csv",
    "Исходный датасет": PROJECT_ROOT / "data" / "movies.csv",
    "Production pipeline": PROJECT_ROOT / "artifacts" / "movie_revenue_pipeline.joblib",
    "Информация о модели": PROJECT_ROOT / "artifacts" / "model_info.json",
    "Метрики модели": PROJECT_ROOT / "artifacts" / "metrics.json",
    "Backend": PROJECT_ROOT / "backend" / "app" / "main.py",
    "Frontend": PROJECT_ROOT / "frontend" / "src" / "App.jsx",
}

pd.DataFrame(
    [{"component": name, "path": str(path.relative_to(PROJECT_ROOT)), "exists": path.exists()} for name, path in important_paths.items()]
)

## Бизнес-постановка

Пользователь вводит параметры фильма, сервис прогнозирует кассовую выручку и рассчитывает бизнес-показатели:

- `predicted_revenue` — прогнозируемая кассовая выручка;
- `predicted_profit = predicted_revenue - budget`;
- `roi = predicted_profit / budget`;
- `payback_ratio = predicted_revenue / budget`;
- `success_category` — категория коммерческой успешности;
- `risk_level` — уровень риска.

In [ ]:
business_formulas = pd.DataFrame([
    {"metric": "predicted_profit", "formula": "predicted_revenue - budget", "meaning": "прогнозируемая прибыль"},
    {"metric": "roi", "formula": "predicted_profit / budget", "meaning": "прибыль относительно бюджета"},
    {"metric": "payback_ratio", "formula": "predicted_revenue / budget", "meaning": "коэффициент кассовой окупаемости"},
])
business_formulas

## Логика категорий

Категории рассчитываются по `roi`, а не по `payback_ratio`:

- `roi < 0` — коммерчески неуспешный;
- `0 <= roi < 1` — умеренно успешный;
- `1 <= roi < 3` — коммерчески успешный;
- `roi >= 3` — высокоуспешный / блокбастер.

In [ ]:
def success_category(roi):
    if roi < 0:
        return "коммерчески неуспешный"
    if roi < 1:
        return "умеренно успешный"
    if roi < 3:
        return "коммерчески успешный"
    return "высокоуспешный / блокбастер"


def risk_level(roi):
    if roi < 0:
        return "высокий"
    if roi < 1:
        return "средний"
    if roi < 3:
        return "низкий"
    return "минимальный"

pd.DataFrame([
    {"roi_example": -0.2, "success_category": success_category(-0.2), "risk_level": risk_level(-0.2)},
    {"roi_example": 0.5, "success_category": success_category(0.5), "risk_level": risk_level(0.5)},
    {"roi_example": 1.8, "success_category": success_category(1.8), "risk_level": risk_level(1.8)},
    {"roi_example": 4.0, "success_category": success_category(4.0), "risk_level": risk_level(4.0)},
])

## Исследовательский маршрут

1. Анализ исходного и очищенного датасета.
2. Подготовка данных и объяснение удаления пропусков.
3. Выбор целевой переменной `gross` и применение `log1p`.
4. Feature engineering.
5. Сравнение моделей.
6. Выбор XGBoost как production-модели.
7. Экспорт единого pipeline.
8. Интеграция через FastAPI.
9. Проверка frontend и end-to-end сценариев.
10. Ограничения и дальнейшая работа с российским датасетом.